In [37]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Exercise01") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [38]:
lax_df = spark.read.csv("data/lax_passengers_header.csv", header=True, inferSchema=True)
lax_df.printSchema()

root
 |-- DataExtractDate: string (nullable = true)
 |-- ReportPeriod: string (nullable = true)
 |-- Terminal: string (nullable = true)
 |-- Arrival_Departure: string (nullable = true)
 |-- Domestic_International: string (nullable = true)
 |-- Passenger_Count: integer (nullable = true)



In [39]:
lax_df.show(5)

+--------------------+--------------------+-----------------+-----------------+----------------------+---------------+
|     DataExtractDate|        ReportPeriod|         Terminal|Arrival_Departure|Domestic_International|Passenger_Count|
+--------------------+--------------------+-----------------+-----------------+----------------------+---------------+
|05/01/2014 12:00:...|01/01/2006 12:00:...|Imperial Terminal|          Arrival|              Domestic|            490|
|05/01/2014 12:00:...|01/01/2006 12:00:...|Imperial Terminal|        Departure|              Domestic|            498|
|05/01/2014 12:00:...|01/01/2006 12:00:...|   Misc. Terminal|          Arrival|              Domestic|            753|
|05/01/2014 12:00:...|01/01/2006 12:00:...|   Misc. Terminal|        Departure|              Domestic|            688|
|05/01/2014 12:00:...|01/01/2006 12:00:...|       Terminal 1|          Arrival|              Domestic|         401535|
+--------------------+--------------------+-----

In [40]:
lax_df.groupBy("ReportPeriod").count().show(truncate=False)

+----------------------+-----+
|ReportPeriod          |count|
+----------------------+-----+
|06/01/2006 12:00:00 AM|36   |
|01/01/2017 12:00:00 AM|38   |
|11/01/2015 12:00:00 AM|37   |
|08/01/2006 12:00:00 AM|34   |
|05/01/2011 12:00:00 AM|39   |
|12/01/2015 12:00:00 AM|38   |
|07/01/2011 12:00:00 AM|34   |
|12/01/2014 12:00:00 AM|35   |
|11/01/2016 12:00:00 AM|38   |
|04/01/2011 12:00:00 AM|37   |
|06/01/2017 12:00:00 AM|38   |
|04/01/2013 12:00:00 AM|36   |
|09/01/2008 12:00:00 AM|35   |
|05/01/2012 12:00:00 AM|35   |
|02/01/2011 12:00:00 AM|37   |
|10/01/2008 12:00:00 AM|35   |
|06/01/2014 12:00:00 AM|36   |
|07/01/2006 12:00:00 AM|38   |
|08/01/2013 12:00:00 AM|38   |
|07/01/2008 12:00:00 AM|37   |
+----------------------+-----+
only showing top 20 rows


In [41]:
from pyspark.sql.functions import col, coalesce, try_to_timestamp, date_format, lit

lax_df = lax_df.withColumn(
    "ReportPeriod",
    date_format(
        coalesce(
            try_to_timestamp(col("ReportPeriod"), lit("MM/dd/yyyy hh:mm:ss a")),
            try_to_timestamp(col("ReportPeriod"), lit("MM/dd/yyyy"))
        ),
        "MM/yyyy"
    )
)

In [42]:
from pyspark.sql.functions import sum as _sum

result = lax_df.where(
    col("Terminal").isin([
        "Terminal 1", "Terminal 2", "Terminal 3", "Terminal 4", "Terminal 5",
        "Terminal 6", "Terminal 7", "Terminal 8", "Tom Bradley International Terminal"
    ])
).groupBy("ReportPeriod") \
    .agg(_sum("Passenger_Count").alias("total_passengers")) \
    .where(col("total_passengers") > 5000000) \
    .orderBy("ReportPeriod")

result.show()

+------------+----------------+
|ReportPeriod|total_passengers|
+------------+----------------+
|     01/2014|         5263473|
|     01/2015|         5339983|
|     01/2016|         5869759|
|     01/2017|         6213892|
|     02/2016|         5280381|
|     02/2017|         5387511|
|     03/2006|         5088556|
|     03/2007|         5240144|
|     03/2008|         5232233|
|     03/2012|         5252277|
|     03/2013|         5385531|
|     03/2014|         5622137|
|     03/2015|         5916774|
|     03/2016|         6277897|
|     03/2017|         6525366|
|     04/2006|         5085946|
|     04/2007|         5172120|
|     04/2011|         5028056|
|     04/2012|         5160973|
|     04/2013|         5157963|
+------------+----------------+
only showing top 20 rows


In [43]:
result.count()

94

In [44]:
lax_df.createOrReplaceTempView("lax_df")

result_sql = spark.sql("""
    SELECT
        ReportPeriod,
        SUM(Passenger_Count) AS total_passengers
    FROM lax_df
    WHERE Terminal IN (
        'Terminal 1', 'Terminal 2', 'Terminal 3', 'Terminal 4', 'Terminal 5',
        'Terminal 6', 'Terminal 7', 'Terminal 8', 'Tom Bradley International Terminal'
    )
    GROUP BY ReportPeriod
    HAVING SUM(Passenger_Count) > 5000000
    ORDER BY ReportPeriod
""")

result_sql.show()

+------------+----------------+
|ReportPeriod|total_passengers|
+------------+----------------+
|     01/2014|         5263473|
|     01/2015|         5339983|
|     01/2016|         5869759|
|     01/2017|         6213892|
|     02/2016|         5280381|
|     02/2017|         5387511|
|     03/2006|         5088556|
|     03/2007|         5240144|
|     03/2008|         5232233|
|     03/2012|         5252277|
|     03/2013|         5385531|
|     03/2014|         5622137|
|     03/2015|         5916774|
|     03/2016|         6277897|
|     03/2017|         6525366|
|     04/2006|         5085946|
|     04/2007|         5172120|
|     04/2011|         5028056|
|     04/2012|         5160973|
|     04/2013|         5157963|
+------------+----------------+
only showing top 20 rows


In [45]:
result_sql.count()

94